# Networks in motion, animated

The `netmotion` package can show its models as they run: a network growing, an infection spreading, opinions changing and a bank failure travelling through a banking system.

Each `animate_*` function returns a Matplotlib animation. `to_html` turns it into a player; press ▶ under it to watch.

Every example starts with its parameters, each on its own line: change one and run the cell again.

In [ ]:
import numpy as np

from netmotion.dynamics import random_initial_states, run_hopfield
from netmotion.finance import InterbankNetwork, bank_capital, four_state_cascade, strategy_growth
from netmotion.metrics import shortest_path_lengths
from netmotion.networks import barabasi_albert, watts_strogatz
from netmotion.visualization import (
    animate_cascade, animate_frames, animate_growth, animate_propagation,
    animate_states, save_animation, to_html,
)

## A network growing

A Barabási–Albert network. Each new node joins a few nodes already there, the better-connected ones more often, so a few hubs pull ahead.

Each node is drawn by its degree at that moment; the newest one is orange.

In [ ]:
n_nodes = 80              # nodes in the finished network
links_per_new_node = 2    # links each new node makes when it arrives
initial_nodes = links_per_new_node + 1   # the starting network, all joined to each other
random_seed = 1           # change it to grow a different network

growing = barabasi_albert(n_nodes=n_nodes, links_per_new_node=links_per_new_node,
                          initial_nodes=initial_nodes, rng=random_seed)
to_html(animate_growth(growing, initial_nodes=initial_nodes))

## An infection spreading

The infection of Section 5.4: at each instant every infected node infects all its neighbours. It runs on a small-world network, a ring whose links are sometimes rewired.

Healthy nodes are green, infected ones red, and the nodes infected at this very instant orange. A link turns red once both of its ends are infected; the long shortcuts carry the infection across the ring.

In [ ]:
n_nodes = 60              # nodes on the ring
neighbours = 4            # each node joined to its 4 nearest neighbours
rewiring = 0.08           # probability that a link is moved to a node at random
first_infected = 0        # the node where the infection starts
random_seed = 1

small_world = watts_strogatz(n_nodes=n_nodes, neighbours=neighbours,
                             rewiring_probability=rewiring, rng=random_seed)
to_html(animate_propagation(small_world, first_infected, layout="circular"))

## Opinions changing

The Hopfield model of Section 5.1 on the same network. Some nodes start in favour (+1, blue) and the rest against (−1, red); at each step every node takes the sign of the sum of its neighbours' states.

In [ ]:
starting_against = 25     # nodes that start at -1; the others start at +1
steps = 8                 # steps of the Hopfield rule
random_seed = 2           # which nodes start against

starting_opinions = random_initial_states(n_nodes, starting_against, rng=random_seed)
opinions = run_hopfield(small_world, starting_opinions, steps=steps)
to_html(animate_states(small_world, opinions))

## A banking crisis

A banking system of several countries, grown bank by bank with the seven strategies of Section 5.6.4. Then one bank fails, the largest by default, and the failure travels along the loans.

Standing banks are brown when they owe more than they lend, green when they lend more than they owe, and purple when the two are equal; they turn red when they fall. Each bank is sized by its balance, and the banks of each country sit together.

In [ ]:
countries = 6             # communities of banks, one per country
new_banks = 120           # banks added one by one after the 3 founding banks of each country
reserves = 20.0           # % of its balance a bank that owes more than it lends keeps as a buffer
reaction_threshold = 30.0 # % of its starting capital a bank can lose before it calls in its loans
random_seed = 7

grown, strategy_of_each_bank = strategy_growth(n_communities=countries, steps=new_banks,
                                               rng=random_seed)
banks = InterbankNetwork.from_community_network(grown)
first_to_fail = banks.largest_bank()

crash = four_state_cascade(banks.network, first_to_fail, bank_capital(banks.network),
                      reserves=reserves, reaction_threshold=reaction_threshold)
print(f"{banks.n_banks} banks, {len(banks.too_big_to_fail())} too big to fail; "
      f"{100 * crash.crashed_fraction:.0f}% fall in {crash.steps} steps")
to_html(animate_cascade(banks, crash))

## Your own animation

`animate_frames` animates anything, given a colour for every node in every frame. Here the nodes of the small-world network light up by their distance from one node, one ring of distance per frame.

In [ ]:
centre = 0                # the node the distances are measured from
lit, unlit = "#e34948", "#d9d9d9"

distance = shortest_path_lengths(small_world, centre)
farthest = int(distance.max())
frames = [np.where(distance <= r, lit, unlit) for r in range(farthest + 1)]
titles = [f"within {r} links of node {centre}" for r in range(farthest + 1)]
to_html(animate_frames(small_world, frames, layout="circular", titles=titles, interval=600))

## Saving

`save_animation` writes an animation to a GIF, an MP4 (if ffmpeg is installed) or a stand-alone HTML page.

In [ ]:
file_name = "cascade.gif"
frames_per_second = 2

path = save_animation(animate_cascade(banks, crash), file_name, fps=frames_per_second)
print("saved", path)

## Try this

- In the banking crisis, change `countries` and `new_banks`, or set `first_to_fail` to another bank than the largest one.
- Raise `reserves` from 20 to 60 and watch more of the system hold.
- Start the infection at the biggest hub of the growing network: `animate_propagation(growing, int(np.argmax(growing.degrees())))`.
- Start the opinions with `starting_against = 50` instead of 25.